In [ ]:
!pip install kagglehub sentence-transformers scikit-learn nltk

In [ ]:
import kagglehub

path = kagglehub.dataset_download("tboyle10/medicaltranscriptions")

print("Dataset path:", path)

In [ ]:
import pandas as pd
import os

file_path = os.path.join(path, "mtsamples.csv")

df = pd.read_csv(file_path)

df = df.dropna(subset=["transcription"])

print(df.head())

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df["transcription"].apply(preprocess)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df["clean_text"].tolist(), show_progress_bar=True)

In [ ]:
from sklearn.cluster import KMeans

num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters, random_state=42)

clusters = kmeans.fit_predict(embeddings)

df["sense_cluster"] = clusters

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

true_labels = df["medical_specialty"].values
clusters = df["sense_cluster"].values

cluster_to_label = {}

for c in np.unique(clusters):
    labels_in_cluster = true_labels[clusters == c]

    # find most frequent label
    values, counts = np.unique(labels_in_cluster, return_counts=True)
    cluster_to_label[c] = values[np.argmax(counts)]

# convert clusters to predicted labels
predicted_labels = [cluster_to_label[c] for c in clusters]

# compute accuracy
sense_accuracy = accuracy_score(true_labels, predicted_labels)

print("Sense Accuracy:", sense_accuracy)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_matrix = cosine_similarity(embeddings)

scores = []

for i in range(len(df)):
    same_cluster = df["sense_cluster"] == df["sense_cluster"].iloc[i]
    idx = np.where(same_cluster)[0]
    if len(idx) > 1:
        scores.append(np.mean(similarity_matrix[i][idx]))

context_matching_score = np.mean(scores)

print("Context Matching Score:", context_matching_score)

In [ ]:
df[["medical_specialty","sense_cluster","transcription"]].head(20)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(reduced_embeddings[:,0],
            reduced_embeddings[:,1],
            c=df["sense_cluster"],
            cmap="viridis",
            s=20)

plt.title("Clinical Text Clusters")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,5))
sns.countplot(x=df["sense_cluster"])

plt.title("Distribution of Clinical Reports Across Clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of Reports")

plt.show()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize and fit TfidfVectorizer on the entire clean_text
# This 'vectorizer' will be used to get word features.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["clean_text"])
terms = vectorizer.get_feature_names_out()

for i in range(num_clusters): # Use num_clusters from previous cell
    # Create a boolean mask for documents belonging to the current cluster
    cluster_mask = (df["sense_cluster"] == i)

    # Get the TF-IDF matrix for documents in this cluster using the boolean mask
    cluster_tfidf_matrix = tfidf_matrix[cluster_mask.to_numpy()]

    # Sum TF-IDF scores for each term across all documents in the cluster
    # This gives an aggregated importance score for each word in the cluster
    sum_tfidf_scores = cluster_tfidf_matrix.sum(axis=0)

    # Convert to a dense array for easier sorting
    sum_tfidf_scores_dense = np.array(sum_tfidf_scores).flatten()

    # Get the indices of the top 10 terms for this cluster
    top_term_indices = sum_tfidf_scores_dense.argsort()[-10:][::-1] # [::-1] to get in descending order

    # Get the actual top 10 terms
    top_words = [terms[idx] for idx in top_term_indices]

    print(f"Cluster {i} keywords:", top_words)